In [250]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [334]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025
week = 1

# Load play-by-play data for the chosen season
def get_weekly_scorers(season, week):
    pbp = nfl.import_pbp_data(years=[season])

    # Filter for regular season, Week 1
    week_pbp = pbp[(pbp['week'] == week)]

    # Keep only touchdown plays
    week_tds = week_pbp[week_pbp['touchdown'] == 1]

    # Count TDs per scorer. Prefer id+name if both available, else fall back to name only
    use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week_tds.columns]
    if use_cols:
        scorers = (
            week_tds.dropna(subset=use_cols)
            .groupby(use_cols)
            .size()
            .reset_index(name='tds')
        )
        if 'td_player_id' in use_cols:
            scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
        else:
            scorers = scorers.rename(columns={'td_player_name': 'player'})
    else:
        # Fallback if td_* columns not present; derive from rusher/receiver
        rush = week_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
        rec = week_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
        rush.columns = ['player_id', 'player']
        rec.columns = ['player_id', 'player']
        both = pd.concat([rush, rec], ignore_index=True)
        scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

    return scorers

    # Show results
scorers = get_weekly_scorers(2025, 1)
scorers.head(50)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0030061,Z.Ertz,1
1,00-0030279,K.Allen,1
2,00-0030506,T.Kelce,1
3,00-0030564,D.Hopkins,1
4,00-0032764,D.Henry,2
5,00-0033288,G.Kittle,1
6,00-0033293,A.Jones,1
7,00-0033553,J.Conner,1
8,00-0033858,J.Smith,1
9,00-0033873,P.Mahomes,1


In [335]:
predictions = pd.read_csv(f'data/predictions_week_{week}.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions.head(50)



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,0.609973,-145.0,0.018136,0.591837
1,00-0039361,Bucky Irving,RB,TB,0.600935,-140.0,0.017602,0.583333
2,00-0038542,Bijan Robinson,RB,ATL,0.589403,-175.0,-0.046961,0.636364
3,00-0036555,Chuba Hubbard,RB,CAR,0.582197,-105.0,0.070001,0.512195
4,00-0034844,Saquon Barkley,RB,PHI,0.577634,-185.0,-0.071488,0.649123
5,00-0039040,De'Von Achane,RB,MIA,0.574298,-140.0,-0.009035,0.583333
6,00-0036223,Jonathan Taylor,RB,IND,0.552116,-180.0,-0.090742,0.642857
7,00-0039139,Jahmyr Gibbs,RB,DET,0.552014,-105.0,0.039819,0.512195
8,00-0037840,Kyren Williams,RB,LA,0.549713,-140.0,-0.033620,0.583333
9,00-0033553,James Conner,RB,ARI,0.547825,-155.0,-0.060018,0.607843


In [336]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False])

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob,tds
0,00-0032764,Derrick Henry,RB,BAL,0.609973,-145.0,0.018136,0.591837,2
1,00-0039361,Bucky Irving,RB,TB,0.600935,-140.0,0.017602,0.583333,1
2,00-0038542,Bijan Robinson,RB,ATL,0.589403,-175.0,-0.046961,0.636364,1
3,00-0036555,Chuba Hubbard,RB,CAR,0.582197,-105.0,0.070001,0.512195,1
4,00-0034844,Saquon Barkley,RB,PHI,0.577634,-185.0,-0.071488,0.649123,1
5,00-0039040,De'Von Achane,RB,MIA,0.574298,-140.0,-0.009035,0.583333,1
6,00-0037840,Kyren Williams,RB,LA,0.549713,-140.0,-0.033620,0.583333,1
7,00-0033553,James Conner,RB,ARI,0.547825,-155.0,-0.060018,0.607843,1
8,00-0035700,Josh Jacobs,RB,GB,0.547748,-160.0,-0.067637,0.615385,1
9,00-0038597,Chase Brown,RB,CIN,0.544633,-150.0,-0.055367,0.600000,1


In [337]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [338]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[predictions['model_edge'] > 0.10] 
ev = ev[ev['price'] <= 400]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
28,00-0038544,Quentin Johnston,WR,LAC,0.468613,370.0,0.255847,0.212766
38,00-0033858,Jonnu Smith,TE,PIT,0.434041,400.0,0.234041,0.200000
55,00-0038117,Wan'Dale Robinson,WR,NYG,0.391703,400.0,0.191703,0.200000
51,00-0039165,Zach Charbonnet,RB,SEA,0.394862,370.0,0.182096,0.212766
12,00-0036912,DeVonta Smith,WR,PHI,0.534621,180.0,0.177478,0.357143
27,00-0034960,Jakobi Meyers,WR,LV,0.471294,225.0,0.163602,0.307692
20,00-0037744,Trey McBride,TE,ARI,0.495164,200.0,0.161831,0.333333
49,00-0036139,Rico Dowdle,RB,CAR,0.399474,320.0,0.161379,0.238095
18,00-0033293,Aaron Jones,RB,MIN,0.505491,165.0,0.128132,0.377358
17,00-0036158,J.K. Dobbins,RB,DEN,0.511576,160.0,0.126961,0.384615


In [339]:
simulate_betting(ev, scorers)

{'bets': 18, 'hits': 7, 'hit_rate': 0.389, 'total_profit': 74.5, 'roi': 0.414}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036912,DeVonta Smith,PHI,WR,180.0,0.534621,0.177478,0,False,-10.0,0.0
1,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.511576,0.126961,1,True,16.0,26.0
2,00-0033293,Aaron Jones,MIN,RB,165.0,0.505491,0.128132,1,True,16.5,26.5
3,00-0035676,A.J. Brown,PHI,WR,160.0,0.496160,0.111544,0,False,-10.0,0.0
4,00-0037744,Trey McBride,ARI,TE,200.0,0.495164,0.161831,0,False,-10.0,0.0
5,00-0030506,Travis Kelce,KC,TE,165.0,0.481134,0.103775,1,True,16.5,26.5
6,00-0034960,Jakobi Meyers,LV,WR,225.0,0.471294,0.163602,0,False,-10.0,0.0
7,00-0038544,Quentin Johnston,LAC,WR,370.0,0.468613,0.255847,2,True,37.0,47.0
8,00-0034827,DJ Moore,CHI,WR,195.0,0.463670,0.124687,0,False,-10.0,0.0
9,00-0036252,Michael Pittman,IND,WR,215.0,0.440139,0.122678,1,True,21.5,31.5


In [ ]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(15)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,0.609973,-145.0,0.018136,0.591837
1,00-0039361,Bucky Irving,RB,TB,0.600935,-140.0,0.017602,0.583333
2,00-0038542,Bijan Robinson,RB,ATL,0.589403,-175.0,-0.046961,0.636364
3,00-0036555,Chuba Hubbard,RB,CAR,0.582197,-105.0,0.070001,0.512195
4,00-0034844,Saquon Barkley,RB,PHI,0.577634,-185.0,-0.071488,0.649123
5,00-0039040,De'Von Achane,RB,MIA,0.574298,-140.0,-0.009035,0.583333
6,00-0036223,Jonathan Taylor,RB,IND,0.552116,-180.0,-0.090742,0.642857
7,00-0039139,Jahmyr Gibbs,RB,DET,0.552014,-105.0,0.039819,0.512195
8,00-0037840,Kyren Williams,RB,LA,0.549713,-140.0,-0.033620,0.583333
9,00-0033553,James Conner,RB,ARI,0.547825,-155.0,-0.060018,0.607843


In [341]:
simulate_betting(top_rb, scorers)


{'bets': 10, 'hits': 8, 'hit_rate': 0.8, 'total_profit': 35.42, 'roi': 0.354}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.609973,0.018136,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.600935,0.017602,1,True,7.142857,17.142857
2,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.589403,-0.046961,1,True,5.714286,15.714286
3,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.582197,0.070001,1,True,9.523810,19.523810
4,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.577634,-0.071488,1,True,5.405405,15.405405
5,00-0039040,De'Von Achane,MIA,RB,-140.0,0.574298,-0.009035,1,True,7.142857,17.142857
6,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.552116,-0.090742,0,False,-10.000000,0.000000
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.552014,0.039819,0,False,-10.000000,0.000000
8,00-0037840,Kyren Williams,LA,RB,-140.0,0.549713,-0.033620,1,True,7.142857,17.142857
9,00-0033553,James Conner,ARI,RB,-155.0,0.547825,-0.060018,1,True,6.451613,16.451613


In [342]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
#top_wr = top_wr[top_wr['model_edge'] > 0.05]
#top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)


{'bets': 15, 'hits': 3, 'hit_rate': 0.2, 'total_profit': -56.5, 'roi': -0.377}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036912,DeVonta Smith,PHI,WR,180.0,0.534621,0.177478,0,False,-10.0,0.0
1,00-0036900,Ja'Marr Chase,CIN,WR,-130.0,0.533280,-0.031938,0,False,-10.0,0.0
2,00-0031408,Mike Evans,TB,WR,110.0,0.515976,0.039785,0,False,-10.0,0.0
3,00-0039893,Brian Thomas Jr.,JAX,WR,130.0,0.514917,0.080135,1,True,13.0,23.0
4,00-0035676,A.J. Brown,PHI,WR,160.0,0.496160,0.111544,0,False,-10.0,0.0
5,00-0039075,Puka Nacua,LA,WR,140.0,0.489732,0.073066,0,False,-10.0,0.0
6,00-0035659,Terry McLaurin,WAS,WR,130.0,0.485333,0.050550,0,False,-10.0,0.0
7,00-0039337,Malik Nabers,NYG,WR,150.0,0.484836,0.084836,0,False,-10.0,0.0
8,00-0034348,Courtland Sutton,DEN,WR,135.0,0.476793,0.051261,1,True,13.5,23.5
9,00-0034960,Jakobi Meyers,LV,WR,225.0,0.471294,0.163602,0,False,-10.0,0.0


In [343]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 26.5, 'roi': 0.53}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0037744,Trey McBride,ARI,TE,200.0,0.495164,0.161831,0,False,-10.0,0.0
1,00-0030506,Travis Kelce,KC,TE,165.0,0.481134,0.103775,1,True,16.5,26.5
2,00-0036894,Pat Freiermuth,PIT,TE,425.0,0.438417,0.247941,0,False,-10.0,0.0
3,00-0033858,Jonnu Smith,PIT,TE,400.0,0.434041,0.234041,1,True,40.0,50.0
4,00-0034753,Mark Andrews,BAL,TE,205.0,0.420391,0.092522,0,False,-10.0,0.0


In [344]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 4, 'hit_rate': 0.8, 'total_profit': 44.5, 'roi': 0.89}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034857,Josh Allen,BUF,QB,-120.0,0.473835,-0.071619,2,True,8.333333,18.333333
1,00-0036389,Jalen Hurts,PHI,QB,-150.0,0.458598,-0.141402,2,True,6.666667,16.666667
2,00-0034796,Lamar Jackson,BAL,QB,205.0,0.416295,0.088426,1,True,20.500000,30.500000
3,00-0035710,Daniel Jones,IND,QB,190.0,0.338868,-0.005960,2,True,19.000000,29.000000
4,00-0039910,Jayden Daniels,WAS,QB,170.0,0.261527,-0.108844,0,False,-10.000000,0.000000


In [365]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 19,
 'hits': 14,
 'hit_rate': 0.737,
 'total_profit': 74.34,
 'roi': 0.391}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.609973,0.018136,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.600935,0.017602,1,True,7.142857,17.142857
2,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.589403,-0.046961,1,True,5.714286,15.714286
3,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.582197,0.070001,1,True,9.523810,19.523810
4,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.577634,-0.071488,1,True,5.405405,15.405405
5,00-0039040,De'Von Achane,MIA,RB,-140.0,0.574298,-0.009035,1,True,7.142857,17.142857
6,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.552116,-0.090742,0,False,-10.000000,0.000000
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.552014,0.039819,0,False,-10.000000,0.000000
8,00-0037840,Kyren Williams,LA,RB,-140.0,0.549713,-0.033620,1,True,7.142857,17.142857
9,00-0033553,James Conner,ARI,RB,-155.0,0.547825,-0.060018,1,True,6.451613,16.451613


In [389]:
top_vegas = predictions.sort_values('market_implied_prob', ascending=False).head(19)
simulate_betting(top_vegas, scorers)

{'bets': 19,
 'hits': 15,
 'hit_rate': 0.789,
 'total_profit': 71.69,
 'roi': 0.377}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.609973,0.018136,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.600935,0.017602,1,True,7.142857,17.142857
2,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.589403,-0.046961,1,True,5.714286,15.714286
3,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.582197,0.070001,1,True,9.523810,19.523810
4,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.577634,-0.071488,1,True,5.405405,15.405405
5,00-0039040,De'Von Achane,MIA,RB,-140.0,0.574298,-0.009035,1,True,7.142857,17.142857
6,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.552116,-0.090742,0,False,-10.000000,0.000000
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.552014,0.039819,0,False,-10.000000,0.000000
8,00-0037840,Kyren Williams,LA,RB,-140.0,0.549713,-0.033620,1,True,7.142857,17.142857
9,00-0033553,James Conner,ARI,RB,-155.0,0.547825,-0.060018,1,True,6.451613,16.451613


In [390]:
vegas_similar = predictions[predictions['model_edge'] < 0.05]
vegas_similar = vegas_similar[vegas_similar['model_edge'] > 0]
vegas_similar = vegas_similar[vegas_similar['price'] < 300]
simulate_betting(vegas_similar, scorers)

{'bets': 22,
 'hits': 7,
 'hit_rate': 0.318,
 'total_profit': -40.96,
 'roi': -0.186}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.609973,0.018136,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.600935,0.017602,1,True,7.142857,17.142857
2,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.552014,0.039819,0,False,-10.000000,0.000000
3,00-0037248,James Cook,BUF,RB,105.0,0.527458,0.039653,1,True,10.500000,20.500000
4,00-0031408,Mike Evans,TB,WR,110.0,0.515976,0.039785,0,False,-10.000000,0.000000
5,00-0037238,Drake London,ATL,WR,125.0,0.460896,0.016452,0,False,-10.000000,0.000000
6,00-0036875,Rhamondre Stevenson,NE,RB,155.0,0.405771,0.013614,0,False,-10.000000,0.000000
7,00-0039384,Tyrone Tracy Jr.,NYG,RB,160.0,0.401989,0.017374,0,False,-10.000000,0.000000
8,00-0039849,Marvin Harrison Jr.,ARI,WR,150.0,0.401601,0.001601,1,True,15.000000,25.000000
9,00-0038555,Tank Bigsby,JAX,RB,155.0,0.392342,0.000186,0,False,-10.000000,0.000000


In [391]:
top_25 = predictions.head(25)
simulate_betting(top_25, scorers)

{'bets': 25, 'hits': 15, 'hit_rate': 0.6, 'total_profit': 40.84, 'roi': 0.163}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.609973,0.018136,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.600935,0.017602,1,True,7.142857,17.142857
2,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.589403,-0.046961,1,True,5.714286,15.714286
3,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.582197,0.070001,1,True,9.523810,19.523810
4,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.577634,-0.071488,1,True,5.405405,15.405405
5,00-0039040,De'Von Achane,MIA,RB,-140.0,0.574298,-0.009035,1,True,7.142857,17.142857
6,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.552116,-0.090742,0,False,-10.000000,0.000000
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.552014,0.039819,0,False,-10.000000,0.000000
8,00-0037840,Kyren Williams,LA,RB,-140.0,0.549713,-0.033620,1,True,7.142857,17.142857
9,00-0033553,James Conner,ARI,RB,-155.0,0.547825,-0.060018,1,True,6.451613,16.451613


In [392]:
week_2 = pd.read_csv('data/predictions_week_2.csv')
week_2.sort_values('market_implied_prob', ascending=False).head(16)




,player_id,player_display_name,position,team,opponent_team,avg_avg_expected_yac,avg_avg_intended_air_yards,avg_avg_separation,avg_avg_time_to_los,avg_avg_yac_above_expectation,...,rush_matchup_value,pass_matchup_value,season,week,depth_chart_rank,predicted_touchdown_probability,merge_name,price,market_implied_prob,model_edge
0,00-0032764,Derrick Henry,RB,BAL,CLE,0.000000,0.000000,0.000000,1.602144,0.000000,...,0.857107,-0.420436,2025.0,1.0,-1.047657,0.603631,derrick henry,-205.0,0.672131,-0.068501
4,00-0035700,Josh Jacobs,RB,GB,WAS,0.000000,0.000000,0.000000,1.061422,0.000000,...,1.176405,-0.416520,2025.0,1.0,-1.047657,0.551611,josh jacobs,-200.0,0.666667,-0.115056
36,00-0033280,Christian McCaffrey,RB,SF,NO,0.000000,0.000000,0.000000,1.026647,0.000000,...,1.023502,4.842685,2025.0,1.0,-1.047657,0.432285,christian mccaffrey,-180.0,0.642857,-0.210572
2,00-0037840,Kyren Williams,RB,LA,TEN,0.000000,0.000000,0.000000,1.529216,0.000000,...,2.438944,-0.283749,2025.0,1.0,-1.047657,0.560748,kyren williams,-175.0,0.636364,-0.075616
8,00-0039139,Jahmyr Gibbs,RB,DET,CHI,0.000000,0.000000,0.000000,0.559129,0.000000,...,1.797469,1.709930,2025.0,1.0,-1.047657,0.544697,jahmyr gibbs,-170.0,0.629630,-0.084933
14,00-0039040,De'Von Achane,RB,MIA,NE,0.000000,0.000000,0.000000,0.806530,0.000000,...,0.655508,1.500415,2025.0,1.0,-1.047657,0.504077,devon achane,-155.0,0.607843,-0.103766
6,00-0038542,Bijan Robinson,RB,ATL,MIN,0.000000,0.000000,0.000000,1.740069,0.000000,...,2.344141,0.188136,2025.0,1.0,-1.047657,0.550620,bijan robinson,-155.0,0.607843,-0.057223
7,00-0033553,James Conner,RB,ARI,CAR,0.000000,0.000000,0.000000,0.998665,0.000000,...,0.498664,0.407134,2025.0,1.0,-1.047657,0.546881,james conner,-155.0,0.607843,-0.060962
5,00-0038597,Chase Brown,RB,CIN,JAX,0.000000,0.000000,0.000000,1.636657,0.000000,...,1.655122,1.710370,2025.0,1.0,-1.047657,0.551502,chase brown,-150.0,0.600000,-0.048498
1,00-0034844,Saquon Barkley,RB,PHI,KC,0.000000,0.000000,0.000000,1.777949,0.000000,...,0.141174,-0.418312,2025.0,1.0,-1.047657,0.580900,saquon barkley,-150.0,0.600000,-0.019100


In [393]:
week_1_2024 = get_weekly_scorers(2024, 18)
week_2_2024 = get_weekly_scorers(2025, 1)

2024 done.
Downcasting floats.
2025 done.
Downcasting floats.


In [394]:
common_values = set(week_1_2024.player) & set(week_2_2024.player)
common_values

{'B.Irving',
 'B.Robinson',
 'C.Sutton',
 'D.Achane',
 'D.Henry',
 'J.Cook',
 'J.Jacobs',
 'J.Smith',
 'M.Harrison',
 'M.Penix',
 'N.Fant',
 'Z.Ertz'}